<a href="https://colab.research.google.com/github/farazaghajani-eng/repowering_flexibility_optimization/blob/main/Comparision_Repower_vs_NewAsset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pulp

In [3]:
!apt-get install -y glpk-utils  # Install GLPK
!pip install pulp  # Install PuLP

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
Suggested packages:
  libiodbc2-dev
The following NEW packages will be installed:
  glpk-utils libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
0 upgraded, 5 newly installed, 0 to remove and 7 not upgraded.
Need to get 625 kB of archives.
After this operation, 2,158 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsuitesparseconfig5 amd64 1:5.10.1+dfsg-4build1 [10.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libamd2 amd64 1:5.10.1+dfsg-4build1 [21.6 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libcolamd2 amd64 1:5.10.1+dfsg-4build1 [18.0 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libglpk40 amd64 5.0-1 [361 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 glpk-uti

In [4]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, GLPK
from pulp import CPLEX_CMD
# Updated Generator Features
generators = [
    {"name": "gen05", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen10", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen11", "RampUp": 70, "RampDown": 70, "Startup": 100, "StartupCost": 100, "Pmax": 300, "Pmin": 100},
    {"name": "gen14", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen19", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen21", "RampUp": 50, "RampDown": 50, "Startup": 50, "StartupCost": 100, "Pmax": 250, "Pmin": 50},
    {"name": "gen23", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen26", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen27", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen28", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen29", "RampUp": 60, "RampDown": 60, "Startup": 80, "StartupCost": 100, "Pmax": 250, "Pmin": 80},
    {"name": "gen37", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen39", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen43", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen44", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen45", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen52", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen53", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
]

storages = [
    {"name": "PSH1", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH2", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH3", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH4", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH5", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH6", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH7", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH8", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH9", "RampUp": 2, "RampDown": 2, "Startup": 100, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
]
# Demand Upper Bound
demand = [
    3291.96, 2358.38, 1726.88, 3255.33,
    4119.83, 4427.13, 3555.79, 3028.29,
    4099.33, 5044.42, 4487.04, 3549.00
]
#wind of 118-bus IEEE
wind = 3

#Calculate the Net_load
net_load = [demand[i] - wind for i in range(len(demand))]
print(net_load)
# Create Optimization Problem
problem = LpProblem("GeneratorOptimization", LpMinimize)

# Decision Variables
power = {

    **{(g["name"], t): LpVariable(f"power_{g['name']}_{t}", 0, g["Pmax"])
       for g in generators for t in range(len(demand))},
    **{(s["name"], t): LpVariable(f"power_{s['name']}_{t}", 0, s["Pmax"])
       for s in storages for t in range(len(demand))}
}


status = {
    **{(g["name"], t): LpVariable(f"status_{g['name']}_{t}", 0, 1, cat="Binary")
    for g in generators for t in range(len(demand))},
     **{(s["name"], t): LpVariable(f"status_{s['name']}_{t}", 0, 1, cat="Binary")
    for s in storages for t in range(len(demand))}
}

#the estimation of Investment cost of gen26,27,28
Investment_cost = 60404925

# Objective Function: Minimize Cost
problem += lpSum(
    power[g["name"], t] * 0.05 + status[g["name"], t] * g["StartupCost"] + Investment_cost


    # The next line ensures t+1 does not cause an Index Error by excluding the last element of the range
    for g in generators for t in range(len(demand))
) + lpSum(
    power[s["name"], t] * 0.1 + status[s["name"], t] * s["StartupCost"]
    for s in storages for t in range(len(demand))
)

# Constraints
for t in range(len(demand)):
    # Meet Demand
    problem += lpSum(power[g["name"], t] for g in generators) + lpSum(power[s["name"], t] for s in storages) >= net_load[t], f"Demand_{t}"

    # Separate loops for generator and storage constraints:
    for g in generators:
        name_1 = g["name"]
        # Enforce generator minimum and maximum if on
        problem += power[name_1, t] <= status[name_1, t] * g["Pmax"], f"MaxPower_{name_1}_{t}"
        problem += power[name_1, t] >= status[name_1, t] * g["Pmin"], f"MinPower_{name_1}_{t}"

        # Ramp-up and ramp-down constraints for generators
        if t > 0:
            problem += power[name_1, t] - power[name_1, t - 1] <= g["RampUp"], f"RampUp_{name_1}_{t}"
            problem += power[name_1, t - 1] - power[name_1, t] <= g["RampDown"], f"RampDown_{name_1}_{t}"

    for s in storages:
        name_2 = s["name"]
        # Enforce storage minimum and maximum if on
        problem += power[name_2, t] <= status[name_2, t] * s["Pmax"], f"MaxPower_{name_2}_{t}"
        problem += power[name_2, t] >= status[name_2, t] * s["Pmin"], f"MinPower_{name_2}_{t}"

        # Ramp-up and ramp-down constraints for storages
        if t > 0:
            problem += power[name_2, t] - power[name_2, t - 1] <= s["RampUp"], f"RampUp_{name_2}_{t}"
            problem += power[name_2, t - 1] - power[name_2, t] <= s["RampDown"], f"RampDown_{name_2}_{t}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for g in generators:
            problem += power[g["name"], t] <= g["Startup"], f"Startup_Constraint_t0_{g['name']}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for s in storages:
            problem += power[s["name"], t] <= s["Startup"], f"Startup_Constraint_t0_{s['name']}"

# Solve Problem
problem.solve(GLPK())
# Get the optimal cost
if LpStatus[problem.status] == "Optimal":
    optimal_cost = problem.objective.value()
    print(f"The optimal cost is: {optimal_cost}")
else:
    print(f"The problem could not be solved optimally. Status: {LpStatus[problem.status]}")
# Output Results
for g in generators:
    for t in range(len(demand)):
        print(f"{g['name']} at time {t}: Power = {power[g['name'], t].varValue}, Status = {status[g['name'], t].varValue}")
for s in storages:
    for t in range(len(demand)):
        print(f"{s['name']} at time {t}: Power = {power[s['name'], t].varValue}, Status = {status[s['name'], t].varValue}")

[3288.96, 2355.38, 1723.88, 3252.33, 4116.83, 4424.13, 3552.79, 3025.29, 4096.33, 5041.42, 4484.04, 3546.0]
The optimal cost is: 13047487631.329994
gen05 at time 0: Power = 0.0, Status = 0
gen05 at time 1: Power = 0.0, Status = 0
gen05 at time 2: Power = 0.0, Status = 0
gen05 at time 3: Power = 0.0, Status = 0
gen05 at time 4: Power = 0.0, Status = 0
gen05 at time 5: Power = 0.0, Status = 0
gen05 at time 6: Power = 0.0, Status = 0
gen05 at time 7: Power = 0.0, Status = 0
gen05 at time 8: Power = 0.0, Status = 0
gen05 at time 9: Power = 0.0, Status = 0
gen05 at time 10: Power = 0.0, Status = 0
gen05 at time 11: Power = 0.0, Status = 0
gen10 at time 0: Power = 100.0, Status = 1
gen10 at time 1: Power = 100.0, Status = 1
gen10 at time 2: Power = 100.0, Status = 1
gen10 at time 3: Power = 100.0, Status = 1
gen10 at time 4: Power = 160.0, Status = 1
gen10 at time 5: Power = 220.0, Status = 1
gen10 at time 6: Power = 160.0, Status = 1
gen10 at time 7: Power = 100.0, Status = 1
gen10 at time 

In [5]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, GLPK
from pulp import CPLEX_CMD
# Updated Generator Features
generators = [
    {"name": "gen05", "RampUp": 100, "RampDown": 100, "Startup": 200, "StartupCost": 120, "Pmax": 450, "Pmin": 100},
    {"name": "gen10", "RampUp": 60, "RampDown": 60, "Startup": 200, "StartupCost": 110, "Pmax": 450, "Pmin": 100},
    {"name": "gen11", "RampUp": 70, "RampDown": 70, "Startup": 200, "StartupCost": 110, "Pmax": 450, "Pmin": 100},
    {"name": "gen14", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen19", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen21", "RampUp": 50, "RampDown": 50, "Startup": 50, "StartupCost": 100, "Pmax": 250, "Pmin": 50},
    {"name": "gen23", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen26", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen37", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen39", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen43", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen44", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen45", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen52", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen53", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
]

storages = [
    {"name": "PSH1", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH2", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH3", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH4", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH5", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH6", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH7", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH8", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH9", "RampUp": 2, "RampDown": 2, "Startup": 100, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
]
# Demand Upper Bound
demand = [3393.5833333333335, 2822.1666666666665, 1489.25, 2903.9166666666665, 3954.0, 4484.75, 3855.3333333333335, 2945.1666666666665, 3731.4166666666665, 5004.166666666667, 4802.0, 3688.3333333333335]


# Create Optimization Problem
problem = LpProblem("GeneratorOptimization", LpMinimize)

# Decision Variables
power = {

    **{(g["name"], t): LpVariable(f"power_{g['name']}_{t}", 0, g["Pmax"])
       for g in generators for t in range(len(demand))},
    **{(s["name"], t): LpVariable(f"power_{s['name']}_{t}", 0, s["Pmax"])
       for s in storages for t in range(len(demand))}
}


status = {
    **{(g["name"], t): LpVariable(f"status_{g['name']}_{t}", 0, 1, cat="Binary")
    for g in generators for t in range(len(demand))},
     **{(s["name"], t): LpVariable(f"status_{s['name']}_{t}", 0, 1, cat="Binary")
    for s in storages for t in range(len(demand))}
}

#the estimation of Rehabilitation cost of gen5,10,11
Rehabilitation_cost = 30202462.5

# Objective Function: Minimize Cost
problem += lpSum(
    power[g["name"], t] * 135 + status[g["name"], t] * g["StartupCost"] + Rehabilitation_cost

    # The next line ensures t+1 does not cause an Index Error by excluding the last element of the range
    for g in generators for t in range(len(demand))
) + lpSum(
    power[s["name"], t] * 62 + status[s["name"], t] * s["StartupCost"]
    for s in storages for t in range(len(demand))
)

# Constraints
for t in range(len(demand)):
    # Meet Demand
    problem += lpSum(power[g["name"], t] for g in generators) + lpSum(power[s["name"], t] for s in storages) >= demand[t], f"Demand_{t}"

    # Separate loops for generator and storage constraints:
    for g in generators:
        name_1 = g["name"]
        # Enforce generator minimum and maximum if on
        problem += power[name_1, t] <= status[name_1, t] * g["Pmax"], f"MaxPower_{name_1}_{t}"
        problem += power[name_1, t] >= status[name_1, t] * g["Pmin"], f"MinPower_{name_1}_{t}"

        # Ramp-up and ramp-down constraints for generators
        if t > 0:
            problem += power[name_1, t] - power[name_1, t - 1] <= g["RampUp"], f"RampUp_{name_1}_{t}"
            problem += power[name_1, t - 1] - power[name_1, t] <= g["RampDown"], f"RampDown_{name_1}_{t}"

    for s in storages:
        name_2 = s["name"]
        # Enforce storage minimum and maximum if on
        problem += power[name_2, t] <= status[name_2, t] * s["Pmax"], f"MaxPower_{name_2}_{t}"
        problem += power[name_2, t] >= status[name_2, t] * s["Pmin"], f"MinPower_{name_2}_{t}"

        # Ramp-up and ramp-down constraints for storages
        if t > 0:
            problem += power[name_2, t] - power[name_2, t - 1] <= s["RampUp"], f"RampUp_{name_2}_{t}"
            problem += power[name_2, t - 1] - power[name_2, t] <= s["RampDown"], f"RampDown_{name_2}_{t}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for g in generators:
            problem += power[g["name"], t] <= g["Startup"], f"Startup_Constraint_t0_{g['name']}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for s in storages:
            problem += power[s["name"], t] <= s["Startup"], f"Startup_Constraint_t0_{s['name']}"

# Solve Problem
problem.solve(GLPK())
# Get the optimal cost
if LpStatus[problem.status] == "Optimal":
    optimal_cost = problem.objective.value()
    print(f"The optimal cost is: {optimal_cost}")
else:
    print(f"The problem could not be solved optimally. Status: {LpStatus[problem.status]}")
# Output Results
for g in generators:
    for t in range(len(demand)):
        print(f"{g['name']} at time {t}: Power = {power[g['name'], t].varValue}, Status = {status[g['name'], t].varValue}")
for s in storages:
    for t in range(len(demand)):
        print(f"{s['name']} at time {t}: Power = {power[s['name'], t].varValue}, Status = {status[s['name'], t].varValue}")

The optimal cost is: 5441083742.25
gen05 at time 0: Power = 200.0, Status = 1
gen05 at time 1: Power = 100.0, Status = 1
gen05 at time 2: Power = 0.0, Status = 0
gen05 at time 3: Power = 0.0, Status = 0
gen05 at time 4: Power = 100.0, Status = 1
gen05 at time 5: Power = 0.0, Status = 0
gen05 at time 6: Power = 0.0, Status = 0
gen05 at time 7: Power = 0.0, Status = 0
gen05 at time 8: Power = 100.0, Status = 1
gen05 at time 9: Power = 200.0, Status = 1
gen05 at time 10: Power = 100.0, Status = 1
gen05 at time 11: Power = 0.0, Status = 0
gen10 at time 0: Power = 173.58333333, Status = 1
gen10 at time 1: Power = 113.58333333, Status = 1
gen10 at time 2: Power = 100.0, Status = 1
gen10 at time 3: Power = 100.0, Status = 1
gen10 at time 4: Power = 160.0, Status = 1
gen10 at time 5: Power = 100.0, Status = 1
gen10 at time 6: Power = 160.0, Status = 1
gen10 at time 7: Power = 100.0, Status = 1
gen10 at time 8: Power = 160.0, Status = 1
gen10 at time 9: Power = 220.0, Status = 1
gen10 at time 1